# Save to a New Table - Analyze NOAA Weather Data in BigQuery

## Activity Overview 

In this notebook, we will query a dataset and save the results into a new table. \
This is a useful skill when the original data source changes continuously and we need to preserve a specific dataset for continued analysis. It’s also valuable when we are dealing with a large dataset and know we’ll be doing more than one analysis using the same subset of data. 

In this scenario, we’re a data analyst at a local news station. We have been tasked with answering questions for meteorologists about the weather. We will work with public data from the </b>National Oceanic and Atmospheric Administration (NOAA)</b>, which has data for the entire United States. This is why we will need to save a subset of the data in a separate table. 

## Objective
Use SQL queries to create new tables when dealing with complex datasets.

## Access the public dataset

For this activity we will need the <b>NOAA weather data from BigQuery’s public datasets</b>. 

In [1]:
from google.cloud import bigquery

print(bigquery.__version__)

3.40.1


In [2]:
client = bigquery.Client()

print(client.project)

myproject001-504709


In [10]:
datasets = list(client.list_datasets())

for dataset in datasets:
    print(dataset.dataset_id)

In [11]:
# Create Dataset
dataset_id = f"{client.project}.portfolio"

dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"   

dataset = client.create_dataset(dataset, exists_ok=True)

print(f"Dataset created: {dataset.full_dataset_id}")

Dataset created: myproject001-504709:portfolio


In [12]:
datasets = list(client.list_datasets())

for dataset in datasets:
    print(dataset.dataset_id)

portfolio


## Querying the data

The meteorologists who we’re working with have asked us to get the temperature, wind speed, and precipitation for stations La Guardia and JFK, for every day in 2020, in descending order by date, and ascending order by Station ID. 

In [3]:
query = """
SELECT
  stn,

  date,
  -- Use the IF function to replace 9999.9 values, which the dataset description explains is the default value when temperature is missing, with NULLs instead.
       IF(

        temp=9999.9,
        NULL,
        temp) AS temperature,

  -- Use the IF function to replace 999.9 values, which the dataset description explains is the default value when wind speed is missing, with NULLs instead.
       IF(

        wdsp="999.9",
        NULL,
        CAST(wdsp AS Float64)) AS wind_speed,

-- Use the IF function to replace 99.99 values, which the dataset description explains is the default value when precipitation is missing, with NULLs instead.

    IF(

       prcp=99.99,
       0,
       prcp) AS precipitation
FROM
  `bigquery-public-data.noaa_gsod.gsod2020`
WHERE
  stn="725030" -- La Guardia

  OR stn="744860" -- JFK
ORDER BY
  date DESC,
  stn ASC
"""
df = client.query(query).to_dataframe()
df

,stn,date,temperature,wind_speed,precipitation
0,725030,2020-12-31,45.8,9.8,0.10
1,744860,2020-12-31,44.1,9.9,0.06
2,725030,2020-12-30,35.2,8.7,0.00
3,744860,2020-12-30,32.5,8.3,0.00
4,725030,2020-12-29,42.1,13.6,0.00
...,...,...,...,...,...
727,744860,2020-01-03,44.1,5.6,0.04
728,725030,2020-01-02,39.8,8.6,0.00
729,744860,2020-01-02,37.9,10.1,0.00
730,725030,2020-01-01,39.8,13.7,0.01


## Save a new table

Assume the meteorologists also asked us a couple questions while they were preparing for the nightly news: They want the average temperature in June 2020 and the average wind_speed in December 2020. 

Instead of rewriting similar, but slightly different, queries over and over again, there is an easier approach: We can save the results from the original query as a table for future queries. 

In order to make this subset of data easier to query from, we can save the table from the weather data into a new dataset. 

In [13]:
query = """
CREATE TABLE portfolio.nyc_weather AS

SELECT
  stn,

  date,
  -- Use the IF function to replace 9999.9 values, which the dataset description explains is the default value when temperature is missing, with NULLs instead.
       IF(

        temp=9999.9,
        NULL,
        temp) AS temperature,

  -- Use the IF function to replace 999.9 values, which the dataset description explains is the default value when wind speed is missing, with NULLs instead.
       IF(

        wdsp="999.9",
        NULL,
        CAST(wdsp AS Float64)) AS wind_speed,

-- Use the IF function to replace 99.99 values, which the dataset description explains is the default value when precipitation is missing, with NULLs instead.

    IF(

       prcp=99.99,
       0,
       prcp) AS precipitation
FROM
  `bigquery-public-data.noaa_gsod.gsod2020`
WHERE
  stn="725030" -- La Guardia

  OR stn="744860" -- JFK
ORDER BY
  date DESC,
  stn ASC;
"""

client.query(query).result()

## Query our new table

Now that we have the subset of this data saved in a new table, we can query it more easily. 

Let's find the average temperature from the meteorologists first question:

In [15]:
query = """
SELECT
    AVG(temperature) AS Avg_Tem
FROM
    portfolio.nyc_weather 
WHERE
    date BETWEEN '2020-06-01' AND '2020-06-30';
"""
df = client.query(query).to_dataframe()
df

,Avg_Tem
0,72.883333


The ability to save our results into a new table is a helpful trick when we know you're only interested in a subset of a larger complex dataset that we plan on querying multiple times, such as the weather data for just La Guardia and JFK. This also helps minimize errors during our analysis.